# 31 · Safely publish validated lightweight results
The source checkout stays on clean main; publication occurs in a disposable clone. Checkpoints remain on Drive.

In [ ]:
MODEL_ID = "rtdetrv2_l"
DATASET_TRACK = "2class"
BENCHMARK_TRACK = "controlled"
PUBLISH_RESULTS = False
DRY_RUN = True
# False runs against session storage, which is DELETED when the
# session ends. Only for smoke runs - never HPO or final training.
USE_GOOGLE_DRIVE = True


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_BRANCH = "main"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

# The only logic a notebook still owns: make `src` importable. Everything after
# this line - Git state, platform detection, paths, dependency policy - lives in
# src/notebook_bootstrap.py so all notebooks behave identically.
_override = os.environ.get("BENCHMARK_REPO_ROOT")
_candidates = (
    [Path(_override).expanduser()]
    if _override
    else [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/aerial-object-detection-benchmark"),
        Path("/kaggle/working/aerial-object-detection-benchmark"),
    ]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in _candidates
        if (candidate / "src" / "notebook_bootstrap.py").is_file()
    ),
    None,
)
if REPO_PATH is None:
    _host = (
        Path("/content")
        if Path("/content").is_dir()
        else Path("/kaggle/working")
        if Path("/kaggle/working").is_dir()
        else None
    )
    if _host is None:
        raise RuntimeError(
            "Run this notebook from the repository, or set BENCHMARK_REPO_ROOT "
            "to an existing clone."
        )
    REPO_PATH = (_host / "aerial-object-detection-benchmark").resolve()
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_PATH)],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))

from src.notebook_bootstrap import bootstrap_notebook

bootstrap = bootstrap_notebook(
    REPO_PATH,
    requirements_file=None,
    use_google_drive=USE_GOOGLE_DRIVE,
    smoke_test=SMOKE_TEST,
)
notebook_environment = bootstrap.environment
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
NOTEBOOK_PLATFORM = notebook_environment.platform
IN_COLAB = NOTEBOOK_PLATFORM == "colab"
IN_KAGGLE = NOTEBOOK_PLATFORM == "kaggle"
print(bootstrap.summary())

from src.config.benchmark_tracks import load_track_config
track_config = load_track_config(REPO_PATH, BENCHMARK_TRACK)
if BENCHMARK_TRACK != "controlled":
    raise RuntimeError("Legacy publisher accepts controlled artifacts only")
